# Q-learning for Cliff-Walking Problem :-

### Q-learning is an off-policy reinforcement learning algorithm that updates its Q-values based on the best possible action in the next state, regardless of what action the agent actually takes. This makes it distinct from on-policy methods like SARSA, because Q-learning learns an optimal policy by considering hypothetical best actions rather than strictly following the agent’s real behavior. It is particularly powerful when the goal is to converge toward the optimal policy, even if the agent’s exploration strategy is imperfect.

## Off-policy learning :-
### The agent learns from the optimal action values (using max(Q[s’, a’])) rather than the actions it actually executes. This allows Q-learning to be more optimistic and sometimes riskier, as seen in CliffWalking where it may cut across the cliff to reach the goal faster, while SARSA tends to learn safer paths.


In [1]:
import gymnasium as gym
import numpy as np
import random


## Q-learning Implementation :-

### Working on 'Cliff Walking Problem' :-
- Same environment: agent starts at the beginning, goal is to reach the target without falling into the cliff.
- If agent steps into the cliff → immediate penalty and reset to start state.

### Key Difference from SARSA
- SARSA (on-policy): Updates Q-values using the action actually taken (including exploration).
- Q-learning (off-policy): Updates Q-values assuming the best possible action (greedy choice), regardless of what the agent actually did.
- This makes Q-learning more optimistic, often converging faster, but sometimes less stable in risky environments like CliffWalking.

### Parameter γ (gamma) – Discount Factor
- Controls how much future rewards are valued.
- High γ (close to 1): agent plans long-term, values reaching the goal more than avoiding immediate penalties.
- Low γ: agent focuses on short-term safety, may avoid cliffs but struggle to reach the goal efficiently.

### Role of Episodes
- Episodes = independent runs until termination.
- More episodes → more exploration, better convergence of Q-values.
- Too few episodes → agent may not learn optimal paths, especially in environments with traps like cliffs.

### Parameter ε (epsilon) – Exploration Strategy
- Used in ε-greedy policy.
- High ε: more random exploration, helps discover safe paths but delays convergence.
- Low ε: more exploitation, faster convergence but risk of suboptimal policy if exploration was insufficient.


In [2]:
# Here we need to define some of these parameters firstly
gamma = 0.99  # gamma means the discount factor
alpha = 0.5  # learning rate
episodes = 500 # here we can also take episodes to be 1000 or 2000 at first
epsilon = 0.1 # means 10% exploration & 90% exploitation in epsilon-greedy policy 


### np.zeros((48, 4)) → Creates a 2D NumPy array filled with zeros.
### Shape (48, 4):
- 48 → Number of states in CliffWalking (the grid is 4 rows × 12 columns = 48 positions).
- 4 → Number of possible actions (up, right, down, left).
### Meaning :- Each row corresponds to a state, and each column corresponds to an action.
## Initially, all Q-values are set to 0 because the agent has no knowledge yet.


In [3]:
# We need this Q-table to find the optimal action :-

# Q-table => stores Q-values

Q = np.zeros((48, 4))

Q 

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],


### Policy --> epsilon-greedy policy
### THis policy will tell what should be the action based on state

### env.action_space → This defines the set of all possible actions the agent can take in the environment.
- For example, in CartPole, the action space is {0, 1} (push left or right).
- In CliffWalking, the action space is {0, 1, 2, 3} (up, right, down, left).
### .sample() → This method randomly selects one valid action from the action space.
### return env.action_space.sample() → Returns a random action to be executed by the agent.

***
- Q[state] → This gives you the row of the Q-table corresponding to the current state.
Example: In CliffWalking, if state = 37, then Q[37] is a vector of 4 values (one for each action: up, right, down, left).
- np.argmax(Q[state]) → Finds the index of the maximum Q-value in that row.
- If Q[37] = [0.1, 0.5, -0.2, 0.3]
- Then np.argmax(Q[37]) = 1 (because 0.5 is the largest value, corresponding to action 1).
- return np.argmax(Q[state]) → Returns the action with the highest estimated value for the current state.
This is the greedy choice (exploitation).


In [4]:
# Policy --> epsilon-greedy policy : state -> action

def epsilon_greedy(state):
    if random.random() < epsilon:  # exploration case 
        return env.action_space.sample()  # return some random action ==> means exploration 
    else:
        return np.argmax(Q[state]) # exploitation step in an ε-greedy policy for SARSA
        

## Q-Learning Implementation :-

### Learning process with rendering :-

In [6]:
# Q-Learning :-

## In this, we will train our agent for multiple episodes 

# Here we firstly need to create the env. for each episodes

# Here we will not render for all 500 episodes, instead we will just do the rendering for every 50th episode here 

for episode in range(episodes):
    render = (episode % 50 == 0)

    # so whenever we want to render, we will create a different env for that
    if render:
        env = gym.make("CliffWalking-v1", render_mode="human")
        print(f"rendering env for episode = {episode+1}/500")
    else:
        env = gym.make("CliffWalking-v1")

    done = False  # it will tell when the episode has ended
    state, _ = env.reset()  # it will give curr state & some additional info which we don't need actually 

    total_reward = 0 # total reward in one episode
    episode_length = 0 # total moves in episode

    while not done:
        # Here always action will be taken based on optimal Q-value from Q-table 
        action = epsilon_greedy(state)  # for any state, action will be decided by epsilon-greedy policy 

        next_state, reward, terminated, truncated, _ = env.step(action)  # Here now agent will take or execute this action in the env & then will return 5 parameters 
        done = terminated or truncated

        # Q-learning update :- in this we update the Q-table
        Q[state, action] += alpha * (reward + gamma * (np.max(Q[next_state]) - Q[state, action]))

        state = next_state

        total_reward += reward
        episode_length += 1

    print(f"episode = {episode+1}/500 : total reward = {total_reward} & episode length = {episode_length}")
    env.close()



rendering env for episode = 1/500


C:\Users\arpit\anaconda3\envs\pytorch_env\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


episode = 1/500 : total reward = -95 & episode length = 95
episode = 2/500 : total reward = -2265 & episode length = 582
episode = 3/500 : total reward = -223 & episode length = 124
episode = 4/500 : total reward = -52 & episode length = 52
episode = 5/500 : total reward = -201 & episode length = 102
episode = 6/500 : total reward = -168 & episode length = 69
episode = 7/500 : total reward = -92 & episode length = 92
episode = 8/500 : total reward = -48 & episode length = 48
episode = 9/500 : total reward = -29 & episode length = 29
episode = 10/500 : total reward = -71 & episode length = 71
episode = 11/500 : total reward = -90 & episode length = 90
episode = 12/500 : total reward = -20 & episode length = 20
episode = 13/500 : total reward = -141 & episode length = 42
episode = 14/500 : total reward = -53 & episode length = 53
episode = 15/500 : total reward = -151 & episode length = 52
episode = 16/500 : total reward = -19 & episode length = 19
episode = 17/500 : total reward = -180 

***
## What did actually our agent learn? :-

### render_mode="human" :-
- Tells Gymnasium how to display the environment.
- "human" → Opens a visual window (or prints text for toy text environments).
- Other modes include "ansi" (text output for Jupyter), "rgb_array" (returns pixel arrays for custom rendering).


In [10]:
# Here now we want to see the env as humans 

env = gym.make("CliffWalking-v1", render_mode="human")
state, _ = env.reset()
done = False
total_reward = 0
episode_len = 0

while not done:
    # As now agent has already learned, so now no need to choose between exploit & explore
    # We now we can directly select the best optimal action using Q-table
    action = np.argmax(Q[state])
    # next_state, reward, terminated, truncated, _ = env.step(action)
    state, reward, terminated, truncated, _ = env.step(action)
    # Now no need to do :- next_state = state
    done = terminated or truncated 

    # next_state = state   # here we either need to do this or we can simply take the name for next_state return bu env.step() as state only
    
    # Now also no need to do Q-update as that is only done in training process & that is already been done
    total_reward += reward
    episode_len += 1

print(f"total reward = {total_reward} & episode length = {episode_len}")
env.close()
    

total reward = -13 & episode length = 13


## Here this Q-learning algorithm is giving the exact shortest path but not the safest path like SARSA.

In [8]:
Q[36]  # Q(s, a)  state=36


array([ -13.13131313, -114.14121129,  -14.14140991,  -14.14140114])

In [9]:
Q[35] # Q(s, a) state=35

array([-3.03029144, -2.01991804, -1.01010101, -3.0303025 ])

In [11]:
Q[24] # Q(s, a) state=24

array([-13.76087568, -12.12121212, -14.13290277, -13.13131289])